# Location overview

Run from the repository root after building the database from the prepared data bundle; see README.md.


# Initialization and database connection

In [ ]:
from analysis_paths import connect_analysis, data_root, output_path, require_file
import duckdb
import matplotlib.pyplot as plt

# Connect to the prepared DuckDB database
db = connect_analysis()


print("DuckDB version:", db.execute("SELECT version()" ).fetchone()[0])

# Holistic overview

In [ ]:
df_overview = db.sql("SELECT sensor_id, ROUND(10 * LOG10(MEDIAN(median_spla)), 2) as median_dba FROM loudness_per_hour GROUP BY sensor_id ORDER BY sensor_id").df()

# add ir to df_overview
ir_per_hour = db.sql("SELECT * from ir_per_hour").df()
ir_per_sensor = ir_per_hour.groupby('sensor_id')['intermittency_ratio'].mean().reset_index().sort_values('sensor_id', ascending=False)
df_overview = df_overview.merge(ir_per_sensor, on='sensor_id', how='left')

# Add ratio of traffic for 60 dBA and above
pct_traffic60 = db.sql("""
  SELECT
    sensor_id,
    ROUND(100.0 * AVG((vehicle > 0.5)::DOUBLE), 2) AS pct_vehicle60
  FROM sources
  WHERE avg_dba >= 60.0
  GROUP BY sensor_id
  ORDER BY sensor_id ASC
""").df()

# add pct_traffic60 to df_overview by sensor_id
df_overview = df_overview.merge(pct_traffic60, on='sensor_id', how='left')

pct_others = db.sql("""
  SELECT
    sensor_id,
    ROUND(100.0 * AVG((human > 0.5)::DOUBLE), 2) AS pct_human,
    ROUND(100.0 * AVG((bird > 0.5)::DOUBLE), 2) AS pct_bird,
    ROUND(100.0 * AVG((church > 0.5)::DOUBLE), 2) AS pct_church
  FROM sources
  GROUP BY sensor_id
  ORDER BY sensor_id ASC
""").df()

# add pct_others to df_overview by sensor_id
df_overview = df_overview.merge(pct_others, on='sensor_id', how='left')


df_overview.set_index('sensor_id', inplace=True)

df_overview.style.background_gradient(axis=0, cmap='viridis')

# Generate Day-night-cycle figures per sensor
Saved under "figures/hours/"

In [ ]:
hours_output = output_path("figures/hours")
hours_output.mkdir(parents=True, exist_ok=True)
# Day night cycle figure for table per sensor
from PIL import Image

def plot_day_night_cycle(sensor_id: str):
  df = db.sql(f"""
    SELECT
      hour,
      median_dba
    FROM loudness_per_hour
    WHERE sensor_id = '{sensor_id}'
    ORDER BY hour
  """).df()

  colors = []
  for h in df['hour'].astype(int):
      if 7 <= h <= 18:
          colors.append("#fde04c")          # day
      elif 19 <= h <= 22:
          colors.append("#d8902b")           # evening
      else:
          colors.append('#a15ac4')        # night (23, 0–6)

  plt.figure(figsize=(6, 1.5))
  plt.bar(df['hour'], df['median_dba'], color=colors)

  plt.xticks([])
  plt.yticks([])
  plt.gca().spines['top'].set_visible(False)
  plt.gca().spines['right'].set_visible(False)
  plt.gca().spines['left'].set_visible(False)
  plt.gca().spines['bottom'].set_visible(False)
  plt.margins(0.01)
  plt.ylim(30, 70)
  # save
  plt.savefig(f"{hours_output}/{sensor_id}_day_night_cycle.png", bbox_inches='tight', pad_inches=0, dpi=200, transparent=True)
  # open image and downscale to 1/3
  img = Image.open(f"{hours_output}/{sensor_id}_day_night_cycle.png")
  img = img.resize((img.width // 3, img.height // 3), Image.Resampling.BICUBIC)
  img.save(f"{hours_output}/{sensor_id}_day_night_cycle.png")
  plt.close()
  
# generate for all sensors
sensor_ids = db.sql("SELECT DISTINCT sensor_id FROM loudness_per_hour").df()['sensor_id'].tolist()
for sensor_id in sensor_ids:
  plot_day_night_cycle(sensor_id)
  
# Generate overall day night cycle figure for all sensors
df = db.sql(f"""
  SELECT
    hour,
    MEDIAN(median_dba) as median_dba
  FROM loudness_per_hour
  GROUP BY hour
  ORDER BY hour
""").df()

colors = []
for h in df['hour'].astype(int):
    if 7 <= h <= 18:
        colors.append("#fde04c")          # day
    elif 19 <= h <= 22:
        colors.append("#d8902b")           # evening
    else:
        colors.append('#a15ac4')        # night (23, 0–6)

plt.figure(figsize=(6, 1.5))
plt.bar(df['hour'], df['median_dba'], color=colors)

plt.xticks([])
plt.yticks([])
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['left'].set_visible(False)
plt.gca().spines['bottom'].set_visible(False)
plt.margins(0.01)
plt.ylim(30, 70)
# save
plt.savefig(f"{hours_output}/overall_day_night_cycle.png", bbox_inches='tight', pad_inches=0, dpi=200, transparent=True)
# open image and downscale to 1/3
img = Image.open(f"{hours_output}/overall_day_night_cycle.png")
img = img.resize((img.width // 3, img.height // 3), Image.Resampling.BICUBIC)
img.save(f"{hours_output}/overall_day_night_cycle.png")
plt.close()